# Hudson Link: Process Replay & Schedule

Generates every table under `data/processed/` that `analysis.ipynb` reads. Unifies a friend's recording of the Hudson Link real-time feed (CoachUSA tracker, polled about once a minute; see `AGENTS.md` for what it is) into flat tables, and brings in the 511NY GTFS schedule the recording was matched against.

Raw inputs: `data/raw/replay/hudsonlink-YYYY-MM-DD.json` (one file per local day) and `data/raw/gtfs_schedule/*.txt`. The original tooling (`analyze.py`, `viewer.html`, notes) sits in `reference/` and is not used here.

In [ ]:
import json
import zipfile
from io import BytesIO
from pathlib import Path

import pandas as pd
import requests

TZ = "America/New_York"
RAW_REPLAY = Path("data/raw/replay")
RAW_GTFS = Path("data/raw/gtfs_schedule")
PROCESSED = Path("data/processed")

## Unify the replay JSON

Each day file holds one **frame** per poll cycle (`t`, `err`), and each frame lists the **vehicles** on a matched trip. A vehicle row carries its GPS fix, the source's per-stop predictions (`ours`), the raw CoachUSA records they came from (`raw`), and the gtfs.zone position-based estimate (`est`). Side dictionaries hold the trip schedules, stops, `actual` stop times inferred from that day's GPS, and `flags`.

The loader flattens these into five tables, all with UTC epoch seconds converted to New York local time:

| table | grain | what it's for |
|---|---|---|
| `positions` | frame × vehicle | **the unified table**: where each bus was, which trip, delay per source & gtfs.zone |
| `predictions` | frame × vehicle × remaining stop | prediction quality vs `actual` |
| `stop_events` | trip × vehicle × stop | schedule adherence (scheduled vs actual arrival/departure) |
| `frames` | poll cycle | feed uptime / upstream errors / unresolved vehicles |
| `flags` | flagged episode | recorder's own issue detector (jump / behind / diverge / shared / unresolved) |

`service_date` is the replay file's local date (the day the frame was recorded), while `start_date` is the GTFS service day of the trip. They differ for trips running past midnight.

In [ ]:
def to_local(s):
    return pd.to_datetime(s, unit="s", utc=True).dt.tz_convert(TZ)


def load_day(path):
    d = json.loads(path.read_text())
    day = d["date"]
    trips, stops = d["trips"], d["stops"]

    frames, pos, preds, flags = [], [], [], []
    for fr in d["frames"]:
        t = fr["t"]
        frames.append({"service_date": day, "frame_t": t, "err": fr["err"],
                       "n_vehicles": len(fr["v"]), "n_unresolved": len(fr["un"])})
        for v in fr["v"]:
            trip = trips[v["trip"]]
            est_delay, est_next = v["est"] if v["est"] else (None, None)
            pos.append({
                "service_date": day, "frame_t": t,
                "trip_key": v["trip"], "trip_id": trip["trip_id"], "start_date": trip["start_date"],
                "route_id": trip["route"], "vehicle_id": v["id"], "label": v["label"],
                "lat": v["lat"], "lon": v["lon"], "gps_t": v["ts"], "bearing": v["bearing"],
                "source_next_seq": v["next"],
                "source_next_delay_s": v["ours"][0][2] if v["ours"] else None,
                "est_next_seq": est_next, "est_delay_s": est_delay,
                "n_predicted_stops": len(v["ours"]),
            })
            # raw CoachUSA record per stop_id: [stop_id, stop_code, sched, est, act]
            raw = {}
            for r in v["raw"]:
                if r[0] is not None:
                    raw.setdefault(r[0], r)
            sched = {s[0]: (s[1], s[2]) for s in trip["schedule"]}
            for seq, pred_t, delay_s in v["ours"]:
                stop_id, sched_t = sched.get(seq, (None, None))
                r = raw.get(stop_id)
                preds.append({
                    "service_date": day, "frame_t": t, "trip_key": v["trip"], "vehicle_id": v["id"],
                    "route_id": trip["route"], "stop_sequence": seq, "stop_id": stop_id,
                    "sched_t": sched_t, "source_pred_t": pred_t, "source_delay_s": delay_s,
                    "raw_sched_t": r[2] if r else None, "raw_est_t": r[3] if r else None,
                    "raw_act_t": r[4] if r else None,
                    # gtfs.zone prediction = schedule + estimator delay, only for stops it hasn't passed
                    "est_pred_t": sched_t + est_delay
                    if est_delay is not None and sched_t is not None and seq >= est_next else None,
                })

    # stop_events: full schedule of every trip seen, joined to GPS-derived actuals per vehicle
    act_rows = []
    for key, rows in d["actual"].items():
        trip_id, start_date, vehicle_id = key.split("|", 2)
        for seq, arr, dep, approx in rows:
            act_rows.append({"trip_key": f"{trip_id}|{start_date}", "vehicle_id": vehicle_id,
                             "stop_sequence": seq, "actual_arr_t": arr, "actual_dep_t": dep,
                             "actual_approx": bool(approx)})
    sched_rows = [
        {"service_date": day, "trip_key": k, "trip_id": tr["trip_id"], "start_date": tr["start_date"],
         "route_id": tr["route"], "shape_id": tr["shape_id"], "stop_sequence": seq,
         "stop_id": sid, "stop_name": stops[sid][2], "sched_t": st}
        for k, tr in trips.items() for seq, sid, st in tr["schedule"]
    ]
    events = pd.DataFrame(sched_rows).merge(
        pd.DataFrame(act_rows, columns=["trip_key", "vehicle_id", "stop_sequence",
                                        "actual_arr_t", "actual_dep_t", "actual_approx"]),
        on=["trip_key", "stop_sequence"], how="left")

    for f in d["flags"]:
        trip_id, start_date, vehicle_id = (f["trip"].split("|", 2) + [None, None])[:3] if f.get("trip") else (None,) * 3
        flags.append({"service_date": day, **f, "trip_key": f"{trip_id}|{start_date}" if trip_id else None,
                      "vehicle_id": vehicle_id})

    return (pd.DataFrame(frames), pd.DataFrame(pos), pd.DataFrame(preds), events, pd.DataFrame(flags))


parts = [load_day(p) for p in sorted(RAW_REPLAY.glob("hudsonlink-*.json"))]
frames, positions, predictions, stop_events, flags = (
    pd.concat([p[i] for p in parts], ignore_index=True) for i in range(5)
)
print({name: len(df) for name, df in
       [("frames", frames), ("positions", positions), ("predictions", predictions),
        ("stop_events", stop_events), ("flags", flags)]})

### Types, timestamps and derived columns

A trip that runs past midnight shows up in two day files, so `stop_events` is deduplicated on trip × vehicle × stop, keeping the row that has an actual time. `horizon_s` is how far ahead a prediction was made (actual − frame time); the error columns are prediction − actual (positive = predicted too late, i.e. the bus came earlier than predicted).

In [ ]:
for df in (frames, positions, predictions, stop_events, flags):
    df["service_date"] = pd.to_datetime(df["service_date"]).dt.date

frames["frame_time"] = to_local(frames["frame_t"])
positions["frame_time"] = to_local(positions["frame_t"])
positions["gps_age_s"] = positions["frame_t"] - positions["gps_t"]
predictions["frame_time"] = to_local(predictions["frame_t"])
flags["start_time"] = to_local(flags["t"])
flags["end_time"] = to_local(flags["end"])

# stop_events: one row per trip x vehicle x stop (vehicle is null for stops with no GPS actual)
stop_events = (
    stop_events.sort_values(["actual_arr_t", "actual_dep_t"], na_position="last")
    .drop_duplicates(["trip_key", "vehicle_id", "stop_sequence"])
)
# drop schedule-only rows for a trip that has an observed row at the same stop
observed = stop_events.dropna(subset=["vehicle_id"])[["trip_key", "stop_sequence"]].drop_duplicates()
stop_events = pd.concat([
    stop_events.dropna(subset=["vehicle_id"]),
    stop_events[stop_events["vehicle_id"].isna()]
    .merge(observed, how="left", indicator=True).query("_merge == 'left_only'").drop(columns="_merge")
    .drop_duplicates(["trip_key", "stop_sequence"]),
]).sort_values(["start_date", "trip_id", "vehicle_id", "stop_sequence"]).reset_index(drop=True)
for c in ["sched", "actual_arr", "actual_dep"]:
    stop_events[f"{c}_time"] = to_local(stop_events[f"{c}_t"])
stop_events["arr_delay_s"] = stop_events["actual_arr_t"] - stop_events["sched_t"]
stop_events["dep_delay_s"] = stop_events["actual_dep_t"] - stop_events["sched_t"]

# predictions: attach the GPS actual for the same trip x vehicle x stop
act = stop_events.dropna(subset=["vehicle_id"])[
    ["trip_key", "vehicle_id", "stop_sequence", "actual_arr_t", "actual_dep_t", "actual_approx"]]
predictions = predictions.merge(act, on=["trip_key", "vehicle_id", "stop_sequence"], how="left")
predictions["actual_t"] = predictions["actual_arr_t"].fillna(predictions["actual_dep_t"])
predictions["horizon_s"] = predictions["actual_t"] - predictions["frame_t"]
predictions["source_err_s"] = predictions["source_pred_t"] - predictions["actual_t"]
predictions["est_err_s"] = predictions["est_pred_t"] - predictions["actual_t"]
predictions["source_differs_from_raw"] = (
    predictions["raw_est_t"].notna()
    & (predictions["source_pred_t"] != predictions["raw_act_t"].fillna(predictions["raw_est_t"]))
)

positions.head()

### Save processed tables

In [ ]:
for name, df in [("frames", frames), ("positions", positions), ("predictions", predictions),
                 ("stop_events", stop_events), ("flags", flags)]:
    out = PROCESSED / name / "data.parquet"
    out.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out, index=False)
    print(f"{out}: {len(df):,} rows x {df.shape[1]} cols")

## GTFS schedule (511NY)

The recorder matched against the **511NY** Hudson Link GTFS. Its trip ids (`1371020`, …) are the ones in the replay. The Trillium feed on Mobility Database uses different ids and, per `reference/max-notes.md`, is wrong on Columbus Day and several H03 times, so it isn't used here. The 511NY zip is public on S3 (Mobility Database `mdb-915`). Load the unzipped files if they're cached, otherwise fetch and unzip once.

Caveat: this is the *current* zip, not a snapshot taken during the recording. Its `feed_version` is `S1000138`, its files are dated 2026-06-22, and it is valid 2026-06-01 → 2027-06-28, so it very likely matches what ran in October. The check below confirms it: every replay trip id is in the feed. `scheduled_trips` (one row per GTFS trip × service day the replay covers) is saved for the analysis.

In [ ]:
GTFS_URL = "https://s3.amazonaws.com/datatools-511ny/public/Hudson_Link.zip"

if not (RAW_GTFS / "trips.txt").exists():
    resp = requests.get(GTFS_URL, timeout=60)
    resp.raise_for_status()
    RAW_GTFS.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(BytesIO(resp.content)) as z:
        z.extractall(RAW_GTFS)
    print(f"Fetched and unzipped: {RAW_GTFS}")
else:
    print(f"Loaded cached GTFS: {RAW_GTFS}")

gtfs = {p.stem: pd.read_csv(p, dtype=str) for p in sorted(RAW_GTFS.glob("*.txt"))}
print(gtfs["feed_info"].to_dict("records"))
{k: len(v) for k, v in gtfs.items()}

In [ ]:
def active_service_ids(day):
    """service_ids running on `day` (date) per calendar + calendar_dates."""
    ymd = day.strftime("%Y%m%d")
    cal = gtfs["calendar"]
    on = set(cal.loc[(cal[day.strftime("%A").lower()] == "1")
                     & (cal["start_date"] <= ymd) & (cal["end_date"] >= ymd), "service_id"])
    cd = gtfs["calendar_dates"].query("date == @ymd")
    on |= set(cd.loc[cd["exception_type"] == "1", "service_id"])
    on -= set(cd.loc[cd["exception_type"] == "2", "service_id"])
    return on


trips_gtfs = gtfs["trips"].query("route_id != '999'")  # 999 = TEST ROUTE
days = sorted(pd.to_datetime(positions["start_date"]).dt.date.unique())
scheduled_trips = pd.concat(
    [trips_gtfs[trips_gtfs["service_id"].isin(active_service_ids(d))].assign(start_date=d.strftime("%Y%m%d"))
     for d in days],
    ignore_index=True,
)

seen = set(positions["trip_id"])
print("replay trip ids missing from GTFS:", len(seen - set(gtfs["trips"]["trip_id"])))

out = PROCESSED / "scheduled_trips" / "data.parquet"
out.parent.mkdir(parents=True, exist_ok=True)
scheduled_trips.to_parquet(out, index=False)
print(f"{out}: {len(scheduled_trips):,} rows (one per trip x service day)")